# Experiment 7.3.2 — Affine Probe Bridge

Analysis-only notebook. It reads finalized Exp7.3.2 artifacts and does not train models or submit jobs.


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root(start=Path.cwd()):
    cur = start.resolve()
    for candidate in (cur, *cur.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('Could not find repository root')

REPO_ROOT = find_repo_root()
ROOT = REPO_ROOT / 'notebooks' / 'artifacts' / 'experiment_7_3_2_affine_probe_bridge' / 'affine_probe_bridge_v1'
manifest = json.loads((ROOT / 'manifest.json').read_text())
runs = pd.read_csv(ROOT / 'method_runs.csv')
summary = pd.read_csv(ROOT / 'method_summary.csv')
contrasts = pd.read_csv(ROOT / 'contrast_summary.csv')
p7 = pd.read_csv(ROOT / 'p7_legacy_reproduction.csv')
manifest


## Mean test balanced accuracy by bridge case

P0 is the most deployment-like rate/scale-only/bias-free probe. P7 reproduces the legacy whole-count StandardScaler + intercept probe.


In [ ]:
display_cols = [
    'backbone_objective', 'case', 'aggregation', 'center', 'bias',
    'test_ba_mean', 'test_ba_std',
    'legacy_l2_wholecount_probe_test_ba_mean',
    'test_ba_minus_legacy_pp_mean',
    'effective_intercept_l2_mean',
]
summary[display_cols].sort_values(['backbone_objective', 'case'])


In [ ]:
case_order = [f'P{i}' for i in range(8)]
tmp = summary.copy()
tmp['short_case'] = tmp['case'].str.extract(r'^(P\d)')[0]
for backbone in manifest['backbone_objectives']:
    view = tmp[tmp.backbone_objective == backbone].set_index('short_case').reindex(case_order)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.errorbar(case_order, 100 * view['test_ba_mean'], yerr=100 * view['test_ba_std'], marker='o')
    legacy = 100 * view['legacy_l2_wholecount_probe_test_ba_mean'].iloc[0]
    ax.axhline(legacy, linestyle='--', label='Legacy L2 whole-count probe')
    ax.set_title(f'{backbone.upper()} backbone: bridge test BA')
    ax.set_ylabel('Test balanced accuracy (%)')
    ax.legend()
    plt.show()


## Factor contrasts


In [ ]:
contrasts.sort_values(['backbone_objective', 'contrast'])


## P7 legacy reproduction check

The absolute test-BA delta should be zero or numerically negligible if the bridge protocol reproduces the stored Exp7.3 probe under the same sklearn environment.


In [ ]:
p7


## Effective affine offset

This diagnostic separates explicit intercept magnitude from the offset induced by feature centering.


In [ ]:
affine_cols = [
    'backbone_objective', 'case',
    'explicit_intercept_l2_mean', 'centering_offset_l2_mean',
    'effective_intercept_l2_mean', 'effective_intercept_span_mean',
]
summary[affine_cols].sort_values(['backbone_objective', 'case'])
